# SignTax Transfer Learning

COCO Pretrained Faster R-CNN ResNet101

↓

Load pretrained weights

↓

Replace classification head

↓

Set number of classes

↓

Apply data augmentation

↓

Train on your custom dataset

↓

Fine-tune backbone layers

↓

Evaluate mAP and IoU

In [1]:
%pip install -q torch torchvision pycocotools pillow numpy matplotlib

In [3]:
import os, json, copy, time, shutil
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === GOOGLE COLAB: Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Update this path to match where your 'new_version' folder is in Google Drive
# Example: /content/drive/MyDrive/SynTax/new_version
DATASET_ROOT = "/content/drive/MyDrive/no_resize"  # Change this path to your Google Drive location

# Drive mounts can disconnect under load; cache dataset locally to avoid I/O endpoint errors.
if DATASET_ROOT.startswith("/content/drive/"):
    LOCAL_DATASET_ROOT = "/content/no_resize_cache"
    if not os.path.exists(LOCAL_DATASET_ROOT):
        print("Copying dataset from Google Drive to local runtime disk...")
        shutil.copytree(DATASET_ROOT, LOCAL_DATASET_ROOT)
        print(f"Local dataset cache ready: {LOCAL_DATASET_ROOT}")
    else:
        print(f"Using existing local dataset cache: {LOCAL_DATASET_ROOT}")
    DATASET_ROOT = LOCAL_DATASET_ROOT

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VALID_DIR = os.path.join(DATASET_ROOT, "valid")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

NUM_EPOCHS_HEAD     = 30     # train head only
NUM_EPOCHS_FINETUNE = 30     # fine-tune backbone
BATCH_SIZE          = 2
LR_HEAD             = 0.005
LR_FINETUNE         = 0.0005

print(f"Using device: {DEVICE}")
print(f"Dataset root: {DATASET_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying dataset from Google Drive to local runtime disk...
Local dataset cache ready: /content/no_resize_cache
Using device: cuda
Dataset root: /content/no_resize_cache


### Section 1: Pretrained object detector
- Faster R-CNN
- ResNet101

In [4]:
# ResNet101 FPN backbone pretrained on ImageNet (standard starting point for Faster R-CNN)
backbone = resnet_fpn_backbone(
    backbone_name='resnet101',
    weights='ResNet101_Weights.DEFAULT',  # ImageNet pretrained
    trainable_layers=0                    # freeze all backbone layers initially
)

# Build Faster R-CNN with 91 COCO classes to match pretrained head structure
model = FasterRCNN(backbone, num_classes=91)
print("Faster R-CNN ResNet101 FPN built with pretrained backbone.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:02<00:00, 62.4MB/s]


Faster R-CNN ResNet101 FPN built with pretrained backbone.
Total parameters: 60,695,190


### Section 2: Replace Classification Head & Set Number of Classes

In [5]:
# Read class names from the COCO annotation file
with open(os.path.join(TRAIN_DIR, "_annotations.coco.json")) as f:
    coco_data = json.load(f)

# Build class map: COCO category_id -> contiguous label index
# Index 0 is always reserved for background in torchvision Faster R-CNN
# NEW - filters to keep only "sign" (excludes Sign-again)
categories = sorted(
    [c for c in coco_data["categories"] if c.get("supercategory") != "none"],
    key=lambda c: c["id"]
)
class_names     = ["__background__"] + [c["name"] for c in categories]
NUM_CLASSES     = len(class_names)
cat_id_to_label = {c["id"]: i + 1 for i, c in enumerate(categories)}
label_to_cat_id = {v: k for k, v in cat_id_to_label.items()}

print(f"Classes ({NUM_CLASSES}): {class_names}")

# Replace the box predictor (classification head) with one that matches NUM_CLASSES
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

model.to(DEVICE)
print("Classification head replaced for custom classes.")

Classes (2): ['__background__', 'sign']
Classification head replaced for custom classes.


### Section 3: Data Augmentation & Dataset Loading

In [6]:
def get_transform(train=True):
    transforms = [T.ToTensor()]   # converts PIL Image -> FloatTensor [0, 1]
    if train:
        transforms += [
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
        ]
    return T.Compose(transforms)


def _verify_image_readable(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


class COCODetectionDataset(Dataset):
    def __init__(self, img_dir, ann_file, cat_id_to_label, transforms=None, read_retries=3):
        with open(ann_file) as f:
            data = json.load(f)

        self.img_dir         = img_dir
        self.transforms      = transforms
        self.cat_id_to_label = cat_id_to_label
        self.read_retries    = read_retries

        self.imgs = {img["id"]: img for img in data["images"]}
        # Group annotations by image_id, skip crowd annotations
        self.anns = {}
        for ann in data["annotations"]:
            if ann.get("iscrowd", 0):
                continue
            self.anns.setdefault(ann["image_id"], []).append(ann)

        # Keep images with annotations and readable files only.
        candidate_ids = [img_id for img_id in self.anns if img_id in self.imgs]
        valid_ids = []
        missing_count = 0
        unreadable_count = 0

        for img_id in candidate_ids:
            img_path = os.path.join(self.img_dir, self.imgs[img_id]["file_name"])
            if not os.path.exists(img_path):
                missing_count += 1
                continue
            if not _verify_image_readable(img_path):
                unreadable_count += 1
                continue
            valid_ids.append(img_id)

        self.ids = valid_ids
        print(
            f"{os.path.basename(self.img_dir)} -> usable: {len(self.ids)} "
            f"| missing: {missing_count} | unreadable: {unreadable_count}"
        )

    def __len__(self):
        return len(self.ids)

    def _load_image_with_retry(self, path):
        last_error = None
        for attempt in range(self.read_retries):
            try:
                with Image.open(path) as img:
                    return img.convert("RGB")
            except (OSError, UnidentifiedImageError) as exc:
                last_error = exc
                time.sleep(0.2 * (attempt + 1))
        raise OSError(f"Failed to read image after {self.read_retries} attempts: {path}") from last_error

    def __getitem__(self, idx):
        if not self.ids:
            raise RuntimeError(f"No readable images found in: {self.img_dir}")

        # If one sample becomes inaccessible at runtime, try nearby samples instead of crashing training.
        for step in range(len(self.ids)):
            safe_idx = (idx + step) % len(self.ids)
            img_id = self.ids[safe_idx]
            img_info = self.imgs[img_id]
            img_path = os.path.join(self.img_dir, img_info["file_name"])

            try:
                img = self._load_image_with_retry(img_path)
            except OSError:
                continue

            boxes, labels = [], []
            for ann in self.anns[img_id]:
                x, y, w, h = ann["bbox"]
                if w > 1 and h > 1:
                    boxes.append([x, y, x + w, y + h])           # convert to xyxy
                    labels.append(self.cat_id_to_label[ann["category_id"]])

            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            target = {
                "boxes": boxes,
                "labels": labels,
                "image_id": torch.tensor([img_id]),
            }

            if self.transforms:
                img = self.transforms(img)
            return img, target

        raise RuntimeError(
            f"Unable to load any readable sample for index {idx} from {self.img_dir}. "
            "If using Google Drive, remount it and rebuild datasets/loaders."
        )


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = COCODetectionDataset(
    TRAIN_DIR, os.path.join(TRAIN_DIR, "_annotations.coco.json"),
    cat_id_to_label, transforms=get_transform(train=True)
)
val_dataset = COCODetectionDataset(
    VALID_DIR, os.path.join(VALID_DIR, "_annotations.coco.json"),
    cat_id_to_label, transforms=get_transform(train=False)
)

# Test split is optional — skip gracefully if the annotation file is absent.
_test_ann = os.path.join(TEST_DIR, "_annotations.coco.json")
if os.path.exists(_test_ann):
    test_dataset = COCODetectionDataset(
        TEST_DIR, _test_ann,
        cat_id_to_label, transforms=get_transform(train=False)
    )
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
    print(f"Test split loaded: {len(test_dataset)} images")
else:
    test_dataset = None
    test_loader  = None
    print(f"WARNING: No test annotation file at {_test_ann} — test evaluation will be skipped.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=1,          shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset) if test_dataset else 'N/A'}")

train -> usable: 224 | missing: 0 | unreadable: 0
valid -> usable: 8 | missing: 0 | unreadable: 0
test -> usable: 6 | missing: 0 | unreadable: 0
Test split loaded: 6 images
Train: 224 | Val: 8 | Test: 6


### Section 4: Train on Custom Dataset (Head Only — Backbone Frozen)

In [ ]:
def train_one_epoch(model, optimizer, loader, device):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()
    return total_loss / len(loader)


# Freeze entire backbone — only the classification head is trained
for name, param in model.named_parameters():
    if "backbone" in name:
        param.requires_grad = False

params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR_HEAD, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

print("Training classification head (backbone frozen)...")
head_losses = []
for epoch in range(1, NUM_EPOCHS_HEAD + 1):
    loss = train_one_epoch(model, optimizer, train_loader, DEVICE)
    scheduler.step()
    head_losses.append(loss)
    print(f"  Epoch {epoch}/{NUM_EPOCHS_HEAD}  |  Loss: {loss:.4f}")

torch.save(model.state_dict(), "fasterrcnn_resnet101_head_no_resize.pth")
print("Head training complete — model saved to fasterrcnn_resnet101_head_no_resize.pth")

Training classification head (backbone frozen)...
  Epoch 1/30  |  Loss: 0.1889
  Epoch 2/30  |  Loss: 0.1442
  Epoch 3/30  |  Loss: 0.1444
  Epoch 4/30  |  Loss: 0.1407
  Epoch 5/30  |  Loss: 0.1402
  Epoch 6/30  |  Loss: 0.1399
  Epoch 7/30  |  Loss: 0.1410
  Epoch 8/30  |  Loss: 0.1407
  Epoch 9/30  |  Loss: 0.1386
  Epoch 10/30  |  Loss: 0.1394
  Epoch 11/30  |  Loss: 0.1392
  Epoch 12/30  |  Loss: 0.1405
  Epoch 13/30  |  Loss: 0.1395
  Epoch 14/30  |  Loss: 0.1404
  Epoch 15/30  |  Loss: 0.1403
  Epoch 16/30  |  Loss: 0.1398
  Epoch 17/30  |  Loss: 0.1403
  Epoch 18/30  |  Loss: 0.1406
  Epoch 19/30  |  Loss: 0.1398
  Epoch 20/30  |  Loss: 0.1388
  Epoch 21/30  |  Loss: 0.1408
  Epoch 22/30  |  Loss: 0.1390
  Epoch 23/30  |  Loss: 0.1380
  Epoch 24/30  |  Loss: 0.1390
  Epoch 25/30  |  Loss: 0.1406
  Epoch 26/30  |  Loss: 0.1398
  Epoch 27/30  |  Loss: 0.1390
  Epoch 28/30  |  Loss: 0.1391
  Epoch 29/30  |  Loss: 0.1388
  Epoch 30/30  |  Loss: 0.1412
Head training complete — mode

### Section 5: Fine-tune Backbone Layers

In [8]:
# Unfreeze the last two ResNet stages and the FPN for fine-tuning
unfreeze_keywords = {"layer3", "layer4", "fpn"}
for name, param in model.named_parameters():
    if any(kw in name for kw in unfreeze_keywords):
        param.requires_grad = True

params_ft      = [p for p in model.parameters() if p.requires_grad]
optimizer_ft   = torch.optim.SGD(params_ft, lr=LR_FINETUNE, momentum=0.9, weight_decay=1e-4)
scheduler_ft   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=NUM_EPOCHS_FINETUNE)

print("Fine-tuning backbone (layer3, layer4, FPN unfrozen)...")
finetune_losses = []
for epoch in range(1, NUM_EPOCHS_FINETUNE + 1):
    loss = train_one_epoch(model, optimizer_ft, train_loader, DEVICE)
    scheduler_ft.step()
    finetune_losses.append(loss)
    print(f"  Epoch {epoch}/{NUM_EPOCHS_FINETUNE}  |  Loss: {loss:.4f}")

torch.save(model.state_dict(), "fasterrcnn_resnet101_finetuned_no_resize.pth")
print("Fine-tuning complete — model saved to fasterrcnn_resnet101_finetuned_no_resize.pth")

Fine-tuning backbone (layer3, layer4, FPN unfrozen)...
  Epoch 1/30  |  Loss: 0.1411
  Epoch 2/30  |  Loss: 0.1389
  Epoch 3/30  |  Loss: 0.1401
  Epoch 4/30  |  Loss: 0.1393
  Epoch 5/30  |  Loss: 0.1389
  Epoch 6/30  |  Loss: 0.1354
  Epoch 7/30  |  Loss: 0.1315
  Epoch 8/30  |  Loss: 0.1305
  Epoch 9/30  |  Loss: 0.1274
  Epoch 10/30  |  Loss: 0.1233
  Epoch 11/30  |  Loss: 0.1202
  Epoch 12/30  |  Loss: 0.1149
  Epoch 13/30  |  Loss: 0.1100
  Epoch 14/30  |  Loss: 0.1037
  Epoch 15/30  |  Loss: 0.1005
  Epoch 16/30  |  Loss: 0.0949
  Epoch 17/30  |  Loss: 0.0909
  Epoch 18/30  |  Loss: 0.0892
  Epoch 19/30  |  Loss: 0.0889
  Epoch 20/30  |  Loss: 0.0858
  Epoch 21/30  |  Loss: 0.0845
  Epoch 22/30  |  Loss: 0.0827
  Epoch 23/30  |  Loss: 0.0799
  Epoch 24/30  |  Loss: 0.0818
  Epoch 25/30  |  Loss: 0.0811
  Epoch 26/30  |  Loss: 0.0802
  Epoch 27/30  |  Loss: 0.0787
  Epoch 28/30  |  Loss: 0.0813
  Epoch 29/30  |  Loss: 0.0785
  Epoch 30/30  |  Loss: 0.0789
Fine-tuning complete — m

In [11]:
torch.save(model.state_dict(), "/content/drive/MyDrive/fasterrcnn_resnet101_head_no_resize.pth")

In [10]:
torch.save(model.state_dict(), "/content/drive/MyDrive/fasterrcnn_resnet101_finetuned_no_resize.pth")

In [ ]:
# Unfreeze the last two ResNet stages and the FPN for fine-tuning
unfreeze_keywords = {"layer3", "layer4", "fpn"}
for name, param in model.named_parameters():
    if any(kw in name for kw in unfreeze_keywords):
        param.requires_grad = True

params_ft      = [p for p in model.parameters() if p.requires_grad]
optimizer_ft   = torch.optim.SGD(params_ft, lr=LR_FINETUNE, momentum=0.9, weight_decay=1e-4)
scheduler_ft   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=NUM_EPOCHS_FINETUNE)

print("Fine-tuning backbone (layer3, layer4, FPN unfrozen)...")
finetune_losses = []
for epoch in range(1, NUM_EPOCHS_FINETUNE + 1):
    loss = train_one_epoch(model, optimizer_ft, train_loader, DEVICE)
    scheduler_ft.step()
    finetune_losses.append(loss)
    print(f"  Epoch {epoch}/{NUM_EPOCHS_FINETUNE}  |  Loss: {loss:.4f}")

torch.save(model.state_dict(), "fasterrcnn_resnet101_finetuned.pth")
print("Fine-tuning complete — model saved to fasterrcnn_resnet101_finetuned_no_resize.pth")

Fine-tuning backbone (layer3, layer4, FPN unfrozen)...
  Epoch 1/30  |  Loss: 0.1411
  Epoch 2/30  |  Loss: 0.1389
  Epoch 3/30  |  Loss: 0.1401
  Epoch 4/30  |  Loss: 0.1393
  Epoch 5/30  |  Loss: 0.1389
  Epoch 6/30  |  Loss: 0.1354
  Epoch 7/30  |  Loss: 0.1315
  Epoch 8/30  |  Loss: 0.1305
  Epoch 9/30  |  Loss: 0.1274
  Epoch 10/30  |  Loss: 0.1233
  Epoch 11/30  |  Loss: 0.1202
  Epoch 12/30  |  Loss: 0.1149
  Epoch 13/30  |  Loss: 0.1100
  Epoch 14/30  |  Loss: 0.1037
  Epoch 15/30  |  Loss: 0.1005
  Epoch 16/30  |  Loss: 0.0949
  Epoch 17/30  |  Loss: 0.0909
  Epoch 18/30  |  Loss: 0.0892
  Epoch 19/30  |  Loss: 0.0889
  Epoch 20/30  |  Loss: 0.0858
  Epoch 21/30  |  Loss: 0.0845
  Epoch 22/30  |  Loss: 0.0827
  Epoch 23/30  |  Loss: 0.0799
  Epoch 24/30  |  Loss: 0.0818
  Epoch 25/30  |  Loss: 0.0811
  Epoch 26/30  |  Loss: 0.0802
  Epoch 27/30  |  Loss: 0.0787
  Epoch 28/30  |  Loss: 0.0813
  Epoch 29/30  |  Loss: 0.0785
  Epoch 30/30  |  Loss: 0.0789
Fine-tuning complete — m

### Section 6: Evaluate mAP and IoU

#### Evaluation Metrics
| Metric | Description |
|--------|-------------|
| **mAP** | Mean Average Precision — area under the Precision-Recall curve, averaged across IoU thresholds and classes |
| **Precision** | TP / (TP + FP) — of all predicted boxes, how many are correct |
| **Recall** | TP / (TP + FN) — of all ground-truth boxes, how many were found |
| **IoU** | Intersection over Union — overlap between predicted and ground-truth box |
| **Accuracy** | (TP + TN) / Total — overall correctness (box-level) |
| **F1** | 2 × (Precision × Recall) / (Precision + Recall) — harmonic mean |

In [3]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def compute_iou(box_a, box_b):
    """Compute IoU between two boxes in xyxy format."""
    xA = max(box_a[0], box_b[0]); yA = max(box_a[1], box_b[1])
    xB = min(box_a[2], box_b[2]); yB = min(box_a[3], box_b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def collect_predictions(model, loader, device, score_thresh=0.5):
    """Run inference and return (results_coco_fmt, per_image_stats)."""
    model.eval()
    coco_results = []
    per_image    = []  # list of (pred_boxes, pred_labels, gt_boxes, gt_labels)

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            for target, output in zip(targets, outputs):
                img_id   = target["image_id"].item()
                gt_boxes = target["boxes"].cpu().numpy()
                gt_lbls  = target["labels"].cpu().numpy()

                keep = output["scores"].cpu() >= score_thresh
                pred_boxes  = output["boxes"][keep].cpu().numpy()
                pred_scores = output["scores"][keep].cpu().numpy()
                pred_labels = output["labels"][keep].cpu().numpy()

                for box, score, lbl in zip(pred_boxes, pred_scores, pred_labels):
                    x1, y1, x2, y2 = box.tolist()
                    coco_results.append({
                        "image_id":   img_id,
                        "category_id": label_to_cat_id.get(int(lbl), int(lbl)),
                        "bbox":       [x1, y1, x2 - x1, y2 - y1],
                        "score":      float(score),
                    })

                per_image.append((pred_boxes, pred_labels, gt_boxes, gt_lbls))

    return coco_results, per_image


def compute_detailed_metrics(per_image, iou_thresh=0.5):
    """
    Compute Precision, Recall, Accuracy, F1, and mean IoU
    across all images using a greedy box-matching scheme.
    """
    TP = FP = FN = total_iou = matched = 0

    for pred_boxes, pred_labels, gt_boxes, gt_labels in per_image:
        matched_gt = set()
        for pb, pl in zip(pred_boxes, pred_labels):
            best_iou, best_j = 0.0, -1
            for j, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                if j in matched_gt or pl != gl:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= iou_thresh and best_j >= 0:
                TP += 1
                matched_gt.add(best_j)
                total_iou += best_iou
                matched += 1
            else:
                FP += 1
        FN += len(gt_boxes) - len(matched_gt)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    mean_iou  = total_iou / matched if matched > 0 else 0.0
    # Accuracy: TP / (TP + FP + FN) — detection-level Jaccard index
    accuracy  = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0
    return {"Precision": precision, "Recall": recall, "F1": f1,
            "Accuracy": accuracy, "Mean IoU": mean_iou, "TP": TP, "FP": FP, "FN": FN}


def evaluate_full(model, loader, device, ann_file, score_thresh=0.5):
    coco_results, per_image = collect_predictions(model, loader, device, score_thresh)

    # --- COCO mAP ---
    print("=" * 55)
    print("COCO mAP (via pycocotools)")
    print("=" * 55)
    if coco_results:
        coco_gt = COCO(ann_file)
        coco_dt = coco_gt.loadRes(coco_results)
        coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
        coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()
        map_50_95 = coco_eval.stats[0]
        map_50    = coco_eval.stats[1]
    else:
        print("No detections above threshold — mAP = 0")
        map_50_95 = map_50 = 0.0

    # --- Detailed metrics ---
    metrics = compute_detailed_metrics(per_image, iou_thresh=0.5)
    print("\n" + "=" * 55)
    print(f"Detailed Metrics  (IoU threshold = 0.50, score ≥ {score_thresh})")
    print("=" * 55)
    print(f"  mAP @ [0.50:0.95] : {map_50_95:.4f}")
    print(f"  mAP @ 0.50        : {map_50:.4f}")
    print(f"  Precision         : {metrics['Precision']:.4f}")
    print(f"  Recall            : {metrics['Recall']:.4f}")
    print(f"  F1 Score          : {metrics['F1']:.4f}")
    print(f"  Accuracy          : {metrics['Accuracy']:.4f}")
    print(f"  Mean IoU          : {metrics['Mean IoU']:.4f}")
    print(f"  TP / FP / FN      : {metrics['TP']} / {metrics['FP']} / {metrics['FN']}")
    return {**metrics, "mAP@[0.5:0.95]": map_50_95, "mAP@0.5": map_50}


print("=== Validation Set ===")
val_metrics = evaluate_full(model, val_loader, DEVICE,
                             os.path.join(VALID_DIR, "_annotations.coco.json"))

print("\n=== Test Set ===")
test_metrics = evaluate_full(model, test_loader, DEVICE,
                              os.path.join(TEST_DIR, "_annotations.coco.json"))

=== Validation Set ===


NameError: name 'model' is not defined

### Training Loss Curve

In [ ]:
epochs_head = list(range(1, NUM_EPOCHS_HEAD + 1))
epochs_ft   = list(range(NUM_EPOCHS_HEAD + 1, NUM_EPOCHS_HEAD + NUM_EPOCHS_FINETUNE + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Loss curve ---
axes[0].plot(epochs_head, head_losses,     "b-o", label="Head Training")
axes[0].plot(epochs_ft,   finetune_losses, "r-o", label="Fine-tuning")
axes[0].axvline(x=NUM_EPOCHS_HEAD + 0.5, color="gray", linestyle="--", alpha=0.6, label="Fine-tune start")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss — Faster R-CNN ResNet101")
axes[0].legend(); axes[0].grid(True)

# --- Metrics bar chart ---
metric_keys = ["mAP@[0.5:0.95]", "mAP@0.5", "Precision", "Recall", "F1", "Accuracy", "Mean IoU"]
val_vals  = [val_metrics.get(k, 0)  for k in metric_keys]
test_vals = [test_metrics.get(k, 0) for k in metric_keys]

x = np.arange(len(metric_keys)); w = 0.35
axes[1].bar(x - w/2, val_vals,  w, label="Validation", color="steelblue")
axes[1].bar(x + w/2, test_vals, w, label="Test",       color="salmon")
axes[1].set_xticks(x); axes[1].set_xticklabels(metric_keys, rotation=25, ha="right")
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("Score")
axes[1].set_title("Evaluation Metrics — Val vs Test")
axes[1].legend(); axes[1].grid(axis="y")

plt.tight_layout()
plt.savefig("training_metrics.png", dpi=150)
plt.show()
print("Plot saved to training_metrics.png")

In [ ]:
import shutil

# Save trained models and results to Google Drive
output_dir = "/content/drive/MyDrive/SynTax_Results"
os.makedirs(output_dir, exist_ok=True)

# Copy model files
shutil.copy("fasterrcnn_resnet101_head.pth", f"{output_dir}/fasterrcnn_resnet101_head.pth")
shutil.copy("fasterrcnn_resnet101_finetuned.pth", f"{output_dir}/fasterrcnn_resnet101_finetuned.pth")
shutil.copy("training_metrics.png", f"{output_dir}/training_metrics.png")

print(f"\n✅ Results saved to Google Drive: {output_dir}")
print("   - fasterrcnn_resnet101_head.pth")
print("   - fasterrcnn_resnet101_finetuned.pth")
print("   - training_metrics.png")

# **Test**

In [ ]:
# =========================
# A) Predict on one image + draw bounding boxes
# =========================
import os
import random
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision.transforms as T

def predict_and_draw(model, image_path, device, score_thresh=0.5, label_names=None):
    model.eval()

    img_pil = Image.open(image_path).convert("RGB")
    img_tensor = T.ToTensor()(img_pil).to(device)

    with torch.no_grad():
        output = model([img_tensor])[0]

    boxes = output["boxes"].cpu().numpy()
    scores = output["scores"].cpu().numpy()
    labels = output["labels"].cpu().numpy()

    keep = scores >= score_thresh
    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

    # plot image + boxes
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(img_pil)
    ax.set_title(f"Predictions (score >= {score_thresh})")

    for box, score, lbl in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)

        if label_names is not None and int(lbl) < len(label_names):
            cls_name = label_names[int(lbl)]
        else:
            cls_name = f"class_{int(lbl)}"

        ax.text(
            x1, max(0, y1 - 5),
            f"{cls_name}: {score:.2f}",
            color="black",
            fontsize=10,
            bbox=dict(facecolor="yellow", alpha=0.7, edgecolor="none")
        )

    ax.axis("off")
    plt.show()

    print(f"Detected boxes: {len(boxes)}")
    if len(scores) > 0:
        print(f"Avg score: {float(np.mean(scores)):.4f}")
    else:
        print("No detections above threshold.")

In [ ]:
# =========================
# B) Load checkpoint and test on a random test image
# =========================
# IMPORTANT:
# - Run sections that build model/class mapping first (setup -> section 3)
# - Then load checkpoint below

checkpoint_path = "/content/drive/MyDrive/SynTax_Results/fasterrcnn_resnet101_finetuned.pth"
# หรือถ้าอยู่ใน working dir:
# checkpoint_path = "fasterrcnn_resnet101_finetuned.pth"

assert os.path.exists(checkpoint_path), f"Checkpoint not found: {checkpoint_path}"
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("Loaded:", checkpoint_path)

# choose random image from test folder
test_images = [f for f in os.listdir(TEST_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
assert len(test_images) > 0, "No test images found in TEST_DIR."
sample_img = os.path.join(TEST_DIR, random.choice(test_images))
print("Sample image:", sample_img)

predict_and_draw(
    model=model,
    image_path=sample_img,
    device=DEVICE,
    score_thresh=0.5,
    label_names=class_names
)

In [ ]:
# =========================
# C) Quantitative evaluation on TEST split
# =========================
# ใช้ฟังก์ชัน evaluate_full ที่มีอยู่ใน Section 6
test_metrics = evaluate_full(
    model=model,
    loader=test_loader,
    device=DEVICE,
    ann_file=os.path.join(TEST_DIR, "_annotations.coco.json"),
    score_thresh=0.5
)

print("\nFinal TEST metrics:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

In [ ]:
# =========================
# D) Compare 2 models (head vs finetuned) on TEST split
# =========================
import pandas as pd

checkpoints = {
    "head_only": "/content/drive/MyDrive/SynTax_Results/fasterrcnn_resnet101_head.pth",
    "finetuned": "/content/drive/MyDrive/SynTax_Results/fasterrcnn_resnet101_finetuned.pth",
}

rows = []
for name, ckpt in checkpoints.items():
    if not os.path.exists(ckpt):
        print(f"Skip {name}: not found -> {ckpt}")
        continue

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    m = evaluate_full(
        model=model,
        loader=test_loader,
        device=DEVICE,
        ann_file=os.path.join(TEST_DIR, "_annotations.coco.json"),
        score_thresh=0.5
    )

    row = {"model": name}
    row.update(m)
    rows.append(row)

df = pd.DataFrame(rows)
display(df.sort_values("mAP@[0.5:0.95]", ascending=False))